In [1]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder

In [2]:
# 1. Load the data
train_df = pd.read_csv('train.csv')


In [3]:
# 2. Check the class distribution (to see if it's imbalanced)
print("--- Target Class Distribution ---")
print(train_df['class'].value_counts())
print("\n")

# 3. Prepare Features (X) and Target (y)
# We drop 'id' because it's just an identifier, and 'class' because it's our target.
X = train_df.drop(['id', 'class'], axis=1)
y = train_df['class']

# 4. Encode Categorical Variables
# Tree models in scikit-learn need numbers, not text.
cat_cols = ['spectral_type', 'galaxy_population']
for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])

# 5. Set up Stratified K-Fold Cross-Validation
# We use 5 folds. shuffle=True and random_state=42 ensure reproducibility.
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Initialize the model. class_weight='balanced' is crucial for Balanced Accuracy!
model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced', n_jobs=-1)

cv_scores = []

print("--- Starting 5-Fold Cross-Validation ---")
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    # Split data into training and validation for this fold
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    # Train the model
    model.fit(X_train, y_train)
    
    # Predict on the validation set
    preds = model.predict(X_val)
    
    # Calculate Balanced Accuracy
    score = balanced_accuracy_score(y_val, preds)
    cv_scores.append(score)
    print(f"Fold {fold+1} Balanced Accuracy: {score:.4f}")

# Calculate the mean score across all 5 folds
mean_cv_score = sum(cv_scores) / len(cv_scores)
print(f"\n*** Mean Local CV Balanced Accuracy: {mean_cv_score:.4f} ***")

--- Target Class Distribution ---
class
GALAXY    377480
QSO       117143
STAR       82724
Name: count, dtype: int64


--- Starting 5-Fold Cross-Validation ---
Fold 1 Balanced Accuracy: 0.9427
Fold 2 Balanced Accuracy: 0.9433
Fold 3 Balanced Accuracy: 0.9425
Fold 4 Balanced Accuracy: 0.9426
Fold 5 Balanced Accuracy: 0.9424

*** Mean Local CV Balanced Accuracy: 0.9427 ***
